# Exp 7: positive control

Fine-tuning where every weight is projected back into its INT4 rounding cell of the original model after each step, so INT4 maps it to exactly the quantized original. The idea was to check that the pipeline flags a reversal when one is built in.

CiteSeer, seeds 50-54 (dose selection) and 60-64 (evaluation). Outcome: INT4 undoes the constrained update almost completely, but the update never reaches the retrained model in FP32, so the pre-registered control failed. See `results/exp7_positive_control`.

CPU, 30-50 min.

In [ ]:
# 0. Install (Colab and Kaggle already have PyTorch)
!pip install -q torch_geometric

## 1. Configuration

In [ ]:
import os, json, time, random, shutil, datetime, itertools
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torch_geometric
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.utils import subgraph
from torch_geometric.nn import GCNConv
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from sklearn.metrics import roc_auc_score
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

SMOKE_TEST = False          # True = tiny crash test (results are NOT meaningful)
SAVE_TO_DRIVE = False       # Colab only

if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/exp7_positive_control'
else:
    OUT_DIR = 'exp7_positive_control' if not SMOKE_TEST else 'exp7_smoke_test'
for sub in ['fp32', 'quant', 'checkpoints']: os.makedirs(f'{OUT_DIR}/{sub}', exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CFG = dict(
    dataset='CiteSeer',
    hidden=128, dropout=0.5, lr_train=0.01, weight_decay=5e-4, epochs=200,         # frozen since Experiment 5
    train_frac=0.10, delete_frac=0.10,
    ft_lr=0.005, ft_dose=100,                                                      # Experiment 6 frozen dose
    pc_variants={'PC-full': [1.0, 4], 'PC-half': [0.5, 4], 'PC-none': [0.0, 4],    # [share of weights confined, cell bit-width]
                 'PC-full-INT3': [1.0, 3]},                                        # fallback: wider INT3 cells
    pc_doses=[100, 200, 400, 800], cell_bits=4, cell_margin=0.05,
    n_retrain=5, selection_seeds=[50, 51, 52, 53, 54], evaluation_seeds=[60, 61, 62, 63, 64],
    bits=[8, 4, 3], schemes=['per_channel', 'per_tensor'], scale_modes=['recomputed', 'frozen_original'],
    noise_draws=3, min_seeds_agree=4, utility_tolerance=0.02, collapse_drop=0.05, unstable_rms=0.01,
    validation_min_reversal_fraction=0.5,
)
if SMOKE_TEST:
    CFG.update(epochs=20, pc_doses=[20, 40], ft_dose=20, selection_seeds=[50, 51], evaluation_seeds=[60, 61],
               n_retrain=3, noise_draws=1, min_seeds_agree=1)
    print('SMOKE TEST: tiny settings, results are not meaningful.')

def log(msg):
    line = f'[{datetime.datetime.now():%H:%M:%S}] {msg}'
    print(line, flush=True)
    with open(f'{OUT_DIR}/progress.log', 'a') as f: f.write(line + '\n')

ENV = dict(device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           torch=torch.__version__, torch_geometric=torch_geometric.__version__, smoke_test=SMOKE_TEST,
           started=f'{datetime.datetime.now():%Y-%m-%d %H:%M:%S}')
json.dump(CFG, open(f'{OUT_DIR}/config.json', 'w'), indent=2); json.dump(ENV, open(f'{OUT_DIR}/environment.json', 'w'), indent=2)
log(f'Device: {DEVICE} | results folder: {OUT_DIR}')

## 2. Pre-registration (written and saved before any result of this experiment)

In [ ]:
PREREG = f'''# Experiment 7: positive control for quantization reversal. Pre-registration
Saved: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}, before any result of this experiment. Device: {ENV['device']} ({ENV['gpu']}).

## Why
Experiment 6 (quantization stage, seeds 40-44) classified all 36 configurations as PRESERVED (primary CiteSeer INT4
per-channel recomputed: reversal fraction -1.0 %, 95 % CI -2.9 % to +0.8 %). Experiment 6 is not changed or re-analysed here.
This experiment asks whether the same pipeline detects a reversal that is built in by construction.

## Dataset and fixed settings
{CFG['dataset']} only; 2-layer GCN, hidden {CFG['hidden']}, dropout {CFG['dropout']}, Adam lr {CFG['lr_train']}, weight decay {CFG['weight_decay']},
{CFG['epochs']} epochs; {int(100*CFG['train_frac'])} % training nodes, {int(100*CFG['delete_frac'])} % of them deleted; split code identical to Experiment 6.
Selection seeds {CFG['selection_seeds']}, evaluation seeds {CFG['evaluation_seeds']} (all fresh).

## Models per seed
Original; Retrain-1..{CFG['n_retrain']}; FT (Experiment 6 fine-tuning, lr {CFG['ft_lr']}, {CFG['ft_dose']} steps, unchanged);
positive-control variants, all started from the Original with the FT objective (retained labels, reduced graph, Adam lr {CFG['ft_lr']},
weight decay {CFG['weight_decay']}, dropout on), with projection after every optimizer step:
* biases are held at the Original's values;
* in each row of each weight matrix the entry with the largest |w| is held fixed and every other entry stays strictly
  inside (-max, +max) of that row, so the per-row INT{CFG['cell_bits']} scale cannot change;
* a share p of the remaining weights is confined to its INT{CFG['cell_bits']} per-channel rounding cell of the Original,
  [(c - 0.5 + {CFG['cell_margin']}) s, (c + 0.5 - {CFG['cell_margin']}) s] with c = round(w_O / s), s = row max|w_O| / {2**(CFG['cell_bits']-1)-1}.
  PC-full p = 1, PC-half p = 0.5 (random subset per seed), PC-none p = 0.
* Fallback PC-full-INT3: p = 1 with INT3 cells (about 2.3x wider), used for validation only if PC-full fails V1.
For PC-full (PC-full-INT3) this guarantees that INT4 (INT3) per-channel quantization, recomputed or frozen-Original scales,
gives exactly the quantized Original's weights; the notebook asserts this.

## Dose selection (selection seeds only)
Step grid {CFG['pc_doses']}. Band = 2 x sqrt(mean over seeds and Retrain pairs of squared AUC differences) (descriptive tolerance;
the pairs are correlated). For each variant, the smallest dose that passes: mean dist <= band; improvement over Original > 0 on
>= {CFG['min_seeds_agree']} seeds; utility >= Original - {CFG['utility_tolerance']}. If none passes, the largest dose is used and the variant is
labelled "did not reach Retrain in FP32 on the selection seeds". Frozen in frozen_doses.json, never recomputed.

## Evaluation seeds
FP32 gate exactly as in Experiment 6 (improvement CI above 0, >= {CFG['min_seeds_agree']} seeds, within band, utility, Original fails band).
Every model is quantized whether or not it passes (this is a validation experiment; gate status is reported with each result).
Quantization, matched noise, measurements, band_Q / effective band, reversal fraction, excess over noise, node-level
recovery with raw shifts and the instability flag (RMS < {CFG['unstable_rms']}), the verdict order and the clipping label are
identical to the Experiment 6 quantization stage.

## Validation criteria (decided in advance)
V1  PC-full passes the FP32 gate on the evaluation seeds (the built-in forgetting exists).
V2  PC-full at INT{CFG['cell_bits']} per-channel, recomputed AND frozen scales: verdict REVERSAL and the lower 95 % bound of the reversal
    fraction > {CFG['validation_min_reversal_fraction']}. (Expected: about 1, since its quantized weights equal the quantized Original's.)
V3  PC-none at INT{CFG['cell_bits']} per-channel recomputed: not REVERSAL (the extra restrictions alone do not create a reversal).
V4  FT replicates Experiment 6 at INT{CFG['cell_bits']} per-channel recomputed: not REVERSAL.
Fallback: if V1 fails for PC-full, V1 and V2 are applied to PC-full-INT3 at INT3 per-channel instead, and this is reported.
Descriptive (no pass/fail): PC-half reversal fraction; PC-full at INT8 and INT3 and per-tensor; node-level specificity.

## Interpretation
V1-V3 all met: the pipeline can detect a reversal of the size that matters, which supports reading Experiment 6 as a real
negative result for this setting. V2 failed with V1 met: the pipeline lacks power and Experiment 6 must be qualified.
V1 failed: a positive control of this form could not be built; report this, PC-full is still quantized and reported as exploratory.
Scope: the measured membership attack, not formal privacy guarantees. PC-full is an engineered update, not a proposed method.
'''
path = f'{OUT_DIR}/exp7_preregistration.md'
if os.path.exists(path):
    log('Pre-registration already saved earlier: keeping the original file unchanged.')
else:
    open(path, 'w').write(PREREG); log('Pre-registration saved.')
print(open(path).read())

## 3. Model (identical to Experiment 6)

In [ ]:
def norm_adj(edge_index, n):
    ei, w = gcn_norm(edge_index, None, n, add_self_loops=True)
    return torch.sparse_coo_tensor(torch.stack([ei[1], ei[0]]), w, (n, n)).coalesce().to(DEVICE)

class Layer(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.lin = nn.Linear(i, o, bias=False); self.bias = nn.Parameter(torch.zeros(o))
        nn.init.xavier_uniform_(self.lin.weight)
    def forward(self, x, A):
        xw = torch.sparse.mm(x, self.lin.weight.t()) if x.is_sparse else x @ self.lin.weight.t()
        return torch.sparse.mm(A, xw) + self.bias

class GCN(nn.Module):
    def __init__(self, i, h, o, p):
        super().__init__(); self.c1, self.c2, self.p = Layer(i, h), Layer(h, o), p
    def forward(self, xs, A):
        if self.training and self.p > 0:
            xs = torch.sparse_coo_tensor(xs.indices(), F.dropout(xs.values(), self.p, True), xs.shape)
        h = F.relu(self.c1(xs, A)); h = F.dropout(h, self.p, self.training)
        return self.c2(h, A)

# parameter order: c1.bias, c1.lin.weight (2-D), c2.bias, c2.lin.weight (2-D); weights are always selected by shape.
_d = Planetoid('data', 'Cora', transform=T.NormalizeFeatures())[0]
_m = GCN(_d.num_features, 128, 7, 0.5).to(DEVICE).eval()
assert [p.dim() for p in _m.parameters()] == [1, 2, 1, 2]
_r1, _r2 = GCNConv(_d.num_features, 128).to(DEVICE), GCNConv(128, 7).to(DEVICE)
with torch.no_grad():
    _m.c1.bias.add_(0.1)
    _r1.lin.weight.copy_(_m.c1.lin.weight); _r1.bias.copy_(_m.c1.bias); _r2.lin.weight.copy_(_m.c2.lin.weight); _r2.bias.copy_(_m.c2.bias)
    diff = (_m(_d.x.to_sparse().to(DEVICE), norm_adj(_d.edge_index, _d.num_nodes))
            - _r2(F.relu(_r1(_d.x.to(DEVICE), _d.edge_index.to(DEVICE))), _d.edge_index.to(DEVICE))).abs().max().item()
log(f'Equivalence check vs PyG GCNConv: max |difference| = {diff:.2e}')
assert diff < 1e-4, 'implementation does not match GCNConv'

## 4. Quantizer, matched noise, measurements (identical to the Experiment 6 quantization stage)

In [ ]:
def quantize(W, bits, scheme, scale_src=None):
    qmax = 2 ** (bits - 1) - 1; out, scales, clipped, total = [], [], 0, 0
    src = W if scale_src is None else scale_src
    for w, s0 in zip(W, src):
        if w.dim() != 2: out.append(w.clone()); scales.append(None); continue
        s = (s0.abs().amax(1, keepdim=True) if scheme == 'per_channel' else s0.abs().max()) / qmax
        s = torch.clamp(s, min=1e-12).expand_as(w)
        q = torch.round(w / s); clipped += int((q.abs() > qmax).sum()); total += q.numel()
        out.append(torch.clamp(q, -qmax, qmax) * s); scales.append(s)
    return out, scales, clipped / total

def matched_noise(W, Wq, scheme, gen):
    out = []
    for w, wq in zip(W, Wq):
        if w.dim() != 2: out.append(w.clone()); continue
        e = (wq - w).cpu()
        if scheme == 'per_channel':
            idx = torch.argsort(torch.rand(e.shape, generator=gen), dim=1); e = torch.gather(e, 1, idx)
        else:
            e = e.flatten()[torch.randperm(e.numel(), generator=gen)].view_as(e)
        out.append(w + e.to(w.device))
    return out

def flat2d(W): return torch.cat([w.flatten() for w in W if w.dim() == 2])
def cos(a, b): return float((a @ b) / (a.norm() * b.norm() + 1e-30))

_w = [torch.tensor([[0.70, -0.35, 0.10], [0.07, 0.02, -0.01]]), torch.zeros(2)]
assert torch.allclose(quantize(_w, 4, 'per_channel')[0][0], torch.tensor([[0.7, -0.4, 0.1], [0.07, 0.02, -0.01]]), atol=1e-6)
assert torch.allclose(quantize(_w, 4, 'per_tensor')[0][0], torch.tensor([[0.7, -0.4, 0.1], [0.1, 0.0, 0.0]]), atol=1e-6)
_q, _s, _c = quantize([_w[0] * 2, _w[1]], 4, 'per_channel', scale_src=_w)
assert abs(_c - 2 / 6) < 1e-9 and torch.allclose(_q[0][0], torch.tensor([0.7, -0.7, 0.2]), atol=1e-6)
_n = matched_noise(_w, quantize(_w, 4, 'per_channel')[0], 'per_channel', torch.Generator().manual_seed(0))
assert torch.allclose((quantize(_w, 4, 'per_channel')[0][0] - _w[0]).sort(1).values, (_n[0] - _w[0]).sort(1).values, atol=1e-7)
log('Quantizer and matched-noise self-checks passed.')

_DATA, _PROBE = {}, {}
def get_data(name):
    if name not in _DATA:
        d = Planetoid('data', name, transform=T.NormalizeFeatures())[0]
        _DATA[name] = dict(d=d, xs=d.x.to_sparse().to(DEVICE), y=d.y.to(DEVICE), y_np=d.y.numpy(),
                           A_full=norm_adj(d.edge_index, d.num_nodes), C=int(d.y.max()) + 1)
    return _DATA[name]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def make_split(ds, seed):
    G = get_data(ds); d = G['d']; N = d.num_nodes
    rng = np.random.default_rng(seed)
    perm = rng.permutation(N); n_tr = int(CFG['train_frac'] * N)
    tr, held = np.sort(perm[:n_tr]), np.sort(perm[n_tr:])
    D = np.sort(rng.choice(tr, int(CFG['delete_frac'] * len(tr)), replace=False)); retained = np.setdiff1d(tr, D)
    hp = rng.permutation(held); H_cal, H_eval = hp[: len(hp) // 2], hp[len(hp) // 2:]
    R_cal = rng.choice(retained, size=min(len(retained), len(H_cal)), replace=False)
    return dict(train=tr, held=held, D=D, retained=retained, H_cal=H_cal, H_eval=H_eval, R_cal=R_cal)

def make_ctx(ds, sp):
    G = get_data(ds); N = G['d'].num_nodes
    keep = torch.ones(N, dtype=torch.bool); keep[torch.from_numpy(sp['D'])] = False
    ei_red, _ = subgraph(keep, G['d'].edge_index, num_nodes=N)
    if ds not in _PROBE: _PROBE[ds] = GCN(G['d'].num_features, CFG['hidden'], G['C'], CFG['dropout']).to(DEVICE)
    return dict(ds=ds, G=G, sp=sp, A_red=norm_adj(ei_red, N), N=N)

def calibrate(cr, ch):
    cand = np.unique(np.r_[cr, ch]); sr, sh = np.sort(cr), np.sort(ch)
    tpr = 1 - np.searchsorted(sr, cand, 'left') / len(sr); tnr = np.searchsorted(sh, cand, 'left') / len(sh)
    return float(cand[int(np.argmax((tpr + tnr) / 2))])

@torch.no_grad()
def measure(ctx, W, thr_fp32=None):
    G, sp, probe = ctx['G'], ctx['sp'], _PROBE[ctx['ds']]
    for p, w in zip(probe.parameters(), W): p.data = w.clone()
    probe.eval(); y = G['y_np']; N = ctx['N']
    conf = F.softmax(probe(G['xs'], G['A_full']), 1).cpu().numpy()[np.arange(N), y]
    pred_red = probe(G['xs'], ctx['A_red']).argmax(1).cpu().numpy()
    D, He, ret = sp['D'], sp['H_eval'], sp['retained']
    auc = roc_auc_score(np.r_[np.ones(len(D)), np.zeros(len(He))], np.r_[conf[D], conf[He]])
    thr = calibrate(conf[sp['R_cal']], conf[sp['H_cal']]); t2 = thr if thr_fp32 is None else thr_fp32
    m = dict(auc=auc, adv=2 * abs(auc - 0.5), threshold=thr,
             tpr_minus_fpr_recal=(conf[D] >= thr).mean() - (conf[He] >= thr).mean(),
             tpr_minus_fpr_fp32_threshold=(conf[D] >= t2).mean() - (conf[He] >= t2).mean(),
             mean_conf_D=float(conf[D].mean()), mean_conf_Heval=float(conf[He].mean()),
             auc_retained_vs_Heval=roc_auc_score(np.r_[np.ones(len(ret)), np.zeros(len(He))], np.r_[conf[ret], conf[He]]),
             test_acc=float((pred_red[sp['held']] == y[sp['held']]).mean()), acc_deleted=float((pred_red[D] == y[D]).mean()))
    return m, conf

def node_recovery(sp, conf_new, conf_ft, conf_o):
    out = {}
    for name, idx in [('D', sp['D']), ('Heval', sp['H_eval'])]:
        t, dlt = conf_o[idx] - conf_ft[idx], conf_new[idx] - conf_ft[idx]; rms = float(np.sqrt(np.mean(t ** 2)))
        out.update({f'node_recovery_{name}': float(dlt @ t / (t @ t + 1e-30)), f'raw_shift_{name}': float(dlt.mean()),
                    f'raw_target_gap_{name}': float(t.mean()), f'target_rms_{name}': rms, f'unstable_{name}': bool(rms < CFG['unstable_rms'])})
    out['raw_shift_D_minus_Heval'] = out['raw_shift_D'] - out['raw_shift_Heval']
    return out

def ci95(x):
    x = np.asarray(x, float); n = len(x); m = x.mean()
    if n < 2: return m, np.nan, np.nan
    h = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n); return m, m - h, m + h

def retrain_band(auc_by_seed):
    cols = [c for c in auc_by_seed.columns if c.startswith('Retrain-')]
    sq = [(auc_by_seed[a] - auc_by_seed[b]) ** 2 for a, b in itertools.combinations(cols, 2)]
    return float(2 * np.sqrt(pd.concat(sq).mean()))

## 5. Positive-control construction: rounding-cell projection
The projection is applied after every optimizer step. The cell test at the end checks, on a real Original, that a PC-full
model pushed to the cell edges quantizes to **exactly** the quantized Original (both scale modes).

In [ ]:
def cell_bounds(W0, share, bits, gen):
    # returns per-parameter (lo, hi) tensors; biases: lo = hi = W0 (frozen)
    qmax = 2 ** (bits - 1) - 1; mg = CFG['cell_margin']; B = []
    for w0 in W0:
        if w0.dim() != 2: B.append((w0.clone(), w0.clone())); continue
        rmax = w0.abs().amax(1, keepdim=True); s = rmax / qmax; c = torch.round(w0 / s)
        inner = rmax * (1 - 1e-6)
        lo, hi = (-inner).expand_as(w0).clone(), inner.expand_as(w0).clone()      # never reach the row maximum
        conf = (torch.rand(w0.shape, generator=gen) < share).to(w0.device) if 0 < share < 1 else torch.full_like(w0, share == 1, dtype=torch.bool)
        lo = torch.where(conf, torch.maximum(lo, (c - 0.5 + mg) * s), lo); hi = torch.where(conf, torch.minimum(hi, (c + 0.5 - mg) * s), hi)
        am = torch.zeros_like(w0, dtype=torch.bool); am[torch.arange(w0.shape[0]), w0.abs().argmax(1)] = True
        lo = torch.where(am, w0, lo); hi = torch.where(am, w0, hi)                # row maximum held fixed
        B.append((lo, hi))
    return B

@torch.no_grad()
def project(model, B):
    for p, (lo, hi) in zip(model.parameters(), B): p.data = torch.minimum(torch.maximum(p.data, lo), hi)

def same_q_pc(Wa, Wb, mode, bits):
    src = Wb if mode == 'frozen_original' else None
    qa = quantize(Wa, bits, 'per_channel', src)[0]; qb = quantize(Wb, bits, 'per_channel')[0]
    return all(torch.equal(a, b) for a, b in zip(qa, qb) if a.dim() == 2)

# cell test on a trained-looking random Original: push every weight to a random cell edge
_g = torch.Generator().manual_seed(1); _W0 = [torch.randn(16, 40, generator=_g) * 0.1, torch.zeros(16)]
for _b in [4, 3]:
    _B = cell_bounds(_W0, 1.0, _b, _g)
    _Wx = [torch.where(torch.rand(lo.shape, generator=_g) < 0.5, lo, hi) for lo, hi in _B]
    assert same_q_pc(_Wx, _W0, 'recomputed', _b) and same_q_pc(_Wx, _W0, 'frozen_original', _b), 'cell construction broken'
assert not same_q_pc([_W0[0] + 0.3 * _W0[0].abs().amax(1, keepdim=True) / 7, _W0[1]], _W0, 'recomputed', 4)   # sensitivity
log('Cell-construction self-check passed: weights at cell edges quantize exactly like the Original.')

## 6. Training one seed (all models) and saving the exact FP32 checkpoint

In [ ]:
DS = CFG['dataset']
def ck_path(seed): return f'{OUT_DIR}/checkpoints/{DS}_seed{seed}.pt'

def train_seed(seed, pc_doses):
    # pc_doses: dict variant -> list of doses to save
    if os.path.exists(ck_path(seed)): log(f'seed {seed}: FP32 checkpoint exists, not retraining'); return
    t0 = time.time(); G = get_data(DS); d, xs, Y, A_full = G['d'], G['xs'], G['y'], G['A_full']
    sp = make_split(DS, seed); ctx = make_ctx(DS, sp); A_red = ctx['A_red']; N = d.num_nodes
    m_full = torch.zeros(N, dtype=torch.bool, device=DEVICE); m_full[torch.from_numpy(sp['train'])] = True
    m_red = torch.zeros(N, dtype=torch.bool, device=DEVICE); m_red[torch.from_numpy(sp['retained'])] = True
    def train(A, mask, init_seed):
        set_seed(init_seed); m = GCN(d.num_features, CFG['hidden'], G['C'], CFG['dropout']).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=CFG['lr_train'], weight_decay=CFG['weight_decay'])
        for _ in range(CFG['epochs']):
            m.train(); opt.zero_grad(); F.cross_entropy(m(xs, A)[mask], Y[mask]).backward(); opt.step()
        return [p.detach().clone() for p in m.parameters()]
    W = {'Original': train(A_full, m_full, 1000 + seed)}
    for k in range(1, CFG['n_retrain'] + 1): W[f'Retrain-{k}'] = train(A_red, m_red, 2000 + 1000 * (k - 1) + seed)
    def finetune(doses, B=None, name='FT'):
        set_seed(5000 + seed + int(CFG['ft_lr'] * 1e4))      # same dropout stream as Experiment 6 FT
        m = GCN(d.num_features, CFG['hidden'], G['C'], CFG['dropout']).to(DEVICE)
        for p, w in zip(m.parameters(), W['Original']): p.data = w.clone()
        opt = torch.optim.Adam(m.parameters(), lr=CFG['ft_lr'], weight_decay=CFG['weight_decay'])
        for step in range(1, max(doses) + 1):
            m.train(); opt.zero_grad(); F.cross_entropy(m(xs, A_red)[m_red], Y[m_red]).backward(); opt.step()
            if B is not None: project(m, B)
            if step in doses: W[f'{name} dose={step}'] = [p.detach().clone() for p in m.parameters()]
    finetune([CFG['ft_dose']])
    for i, (v, (share, bits)) in enumerate(CFG['pc_variants'].items()):
        B = cell_bounds(W['Original'], share, bits, torch.Generator().manual_seed(7000 + 10 * seed + i))
        finetune(pc_doses[v], B, v)
        if share == 1.0:
            for dose in pc_doses[v]:
                for mode in ['recomputed', 'frozen_original']:
                    assert same_q_pc(W[f'{v} dose={dose}'], W['Original'], mode, bits), f'{v} not identical after INT{bits} ({mode})'
    torch.save(dict(weights={k: [w.cpu() for w in v] for k, v in W.items()}, split=sp, seed=seed, env=ENV), ck_path(seed))
    log(f'seed {seed}: trained and saved checkpoint in {time.time()-t0:.0f}s (PC-full INT4 identity asserted)')

def load_ck(seed):
    ck = torch.load(ck_path(seed), weights_only=False)
    return {k: [w.to(DEVICE) for w in v] for k, v in ck['weights'].items()}, ck['split']

def eval_fp32(seed):
    f = f'{OUT_DIR}/fp32/{DS}_seed{seed}.csv'
    if os.path.exists(f): return
    W, sp = load_ck(seed); ctx = make_ctx(DS, sp); rows = []
    for name, Wm in W.items():
        m, _ = measure(ctx, Wm)
        rows.append(dict(dataset=DS, seed=seed, model=name, update_norm=float((flat2d(Wm) - flat2d(W['Original'])).norm()), **m))
    pd.DataFrame(rows).to_csv(f, index=False)

def fp32_table(seeds):
    return pd.concat([pd.read_csv(f'{OUT_DIR}/fp32/{DS}_seed{s}.csv') for s in seeds])

def criteria(a, u, name, band, gate=False):
    k = CFG['min_seeds_agree']; dist_o = (a['Original'] - a['Retrain-1']).abs(); dist = (a[name] - a['Retrain-1']).abs()
    imp = dist_o - dist; m, lo, hi = ci95(imp)
    c = dict(within_band=bool(dist.mean() <= band), improves_on_enough_seeds=bool((imp > 0).sum() >= k),
             utility_ok=bool(u[name].mean() >= u['Original'].mean() - CFG['utility_tolerance']))
    if gate: c.update(ci_above_0=bool(lo > 0), original_fails_band=bool(dist_o.mean() > band and (dist_o > band).sum() >= k))
    return dict(improvement=float(m), ci95=[float(lo), float(hi)], seeds_improving=f'{int((imp > 0).sum())}/{len(imp)}',
                dist=float(dist.mean()), utility=float(u[name].mean()), utility_original=float(u['Original'].mean()),
                criteria=c, passed=all(c.values()))

## 7. Stage A: selection seeds -> freeze one dose per positive-control variant

In [ ]:
for seed in CFG['selection_seeds']:
    train_seed(seed, {v: CFG['pc_doses'] for v in CFG['pc_variants']}); eval_fp32(seed)
FROZEN_PATH = f'{OUT_DIR}/frozen_doses.json'
if os.path.exists(FROZEN_PATH):
    FROZEN = json.load(open(FROZEN_PATH)); log('Frozen doses already exist: NOT recomputed.')
else:
    t = fp32_table(CFG['selection_seeds']); a = t.pivot_table(index='seed', columns='model', values='auc')
    u = t.pivot_table(index='seed', columns='model', values='test_acc'); band = retrain_band(a); FROZEN = dict(band_selection=band)
    for v in CFG['pc_variants']:
        res = {dose: criteria(a, u, f'{v} dose={dose}', band) for dose in CFG['pc_doses']}
        ok = [d for d in CFG['pc_doses'] if res[d]['passed']]
        FROZEN[v] = dict(dose=min(ok) if ok else max(CFG['pc_doses']),
                         status='reached Retrain' if ok else 'did not reach Retrain in FP32 on the selection seeds',
                         per_dose={str(k): r for k, r in res.items()})
    json.dump(FROZEN, open(FROZEN_PATH, 'w'), indent=2); log('Frozen doses saved.')
print(f"Selection band {FROZEN['band_selection']:.4f}")
for v in CFG['pc_variants']:
    print(f"\n{v}: frozen dose {FROZEN[v]['dose']} ({FROZEN[v]['status']})")
    for dose, r in FROZEN[v]['per_dose'].items():
        print(f"  {dose} steps: dist {r['dist']:.4f}, improvement {r['improvement']:.4f}, {r['seeds_improving']} seeds, "
              f"utility {r['utility']:.3f} vs {r['utility_original']:.3f} -> {'passes' if r['passed'] else 'fails'}")

## 8. Stage B: evaluation seeds, FP32 gate (saved before quantization)

In [ ]:
for seed in CFG['evaluation_seeds']:
    train_seed(seed, {v: [FROZEN[v]['dose']] for v in CFG['pc_variants']}); eval_fp32(seed)
TEST = {'FT': f"FT dose={CFG['ft_dose']}", **{v: f"{v} dose={FROZEN[v]['dose']}" for v in CFG['pc_variants']}}
GATE_PATH = f'{OUT_DIR}/fp32_gate.json'
if os.path.exists(GATE_PATH):
    GATE = json.load(open(GATE_PATH)); log('FP32 gate already decided earlier: NOT recomputed.')
else:
    t = fp32_table(CFG['evaluation_seeds']); a = t.pivot_table(index='seed', columns='model', values='auc')
    u = t.pivot_table(index='seed', columns='model', values='test_acc'); band = retrain_band(a)
    GATE = dict(band_fp32=band, dist_original=float((a['Original'] - a['Retrain-1']).abs().mean()),
                models={k: dict(model=n, **criteria(a, u, n, band, gate=True)) for k, n in TEST.items()})
    json.dump(GATE, open(GATE_PATH, 'w'), indent=2); log('FP32 gate saved.')
print(f"Evaluation band {GATE['band_fp32']:.4f} (descriptive tolerance) | dist(Original) {GATE['dist_original']:.3f}")
for k, r in GATE['models'].items():
    print(f"  {r['model']}: dist {r['dist']:.4f}, improvement {r['improvement']:.4f} (CI {r['ci95'][0]:.4f} to {r['ci95'][1]:.4f}), "
          f"{r['seeds_improving']} seeds, utility {r['utility']:.3f} -> {'PASSED' if r['passed'] else 'NOT PASSED'}")

## 9. Quantization phase (copies of the saved checkpoints; every model)

In [ ]:
CONFIGS = list(itertools.product(CFG['bits'], CFG['schemes'], CFG['scale_modes']))
TEST_NAMES = list(TEST.values())

def quant_seed(seed):
    f = f'{OUT_DIR}/quant/{DS}_seed{seed}.csv'
    if os.path.exists(f): log(f'seed {seed}: quantization already done, skipping'); return
    t0 = time.time(); W, sp = load_ck(seed); ctx = make_ctx(DS, sp)
    names = ['Original'] + [f'Retrain-{k}' for k in range(1, CFG['n_retrain'] + 1)] + TEST_NAMES
    fp = {n: measure(ctx, W[n]) for n in names}; base = dict(dataset=DS, seed=seed)
    rows = [dict(**base, model=n, bits=32, scheme='fp32', scale_mode='fp32', condition='fp32', draw=-1, clip_frac=0.0, **fp[n][0]) for n in names]
    for bits, scheme, mode in CONFIGS:
        cfg = dict(bits=bits, scheme=scheme, scale_mode=mode)
        def src(n): return W['Original'] if (mode == 'frozen_original' and not n.startswith('Retrain')) else None
        Q = {n: quantize(W[n], bits, scheme, src(n)) for n in names}
        for n in names:
            Wq, S, clip = Q[n]; m, conf = measure(ctx, Wq, thr_fp32=fp[n][0]['threshold'])
            r = dict(**base, model=n, **cfg, condition='quantized', draw=-1, clip_frac=clip, **m)
            if n in TEST_NAMES:
                d_fp = flat2d(W[n]) - flat2d(W['Original']); d_q = flat2d(Wq) - flat2d(Q['Original'][0])
                s = torch.cat([x.flatten() for x in S if x is not None]); rel = d_fp.abs() / s
                err = flat2d(Wq) - flat2d(W[n]); to_o = flat2d(W['Original']) - flat2d(W[n])
                r.update(codes_identical_to_Q_original=bool(torch.equal(flat2d(Wq), flat2d(Q['Original'][0]))),
                         upd_norm_fp32=float(d_fp.norm()), upd_norm_ratio=float(d_q.norm() / d_fp.norm()), upd_cosine=cos(d_q, d_fp),
                         share_upd_erased=float((d_q == 0).float().mean()), med_upd_over_bin=float(rel.median()),
                         share_upd_above_half_bin=float((rel > 0.5).float().mean()),
                         rounding_pull_to_original=float(err @ to_o / (to_o @ to_o)), rounding_cos_to_original=cos(err, to_o),
                         **node_recovery(sp, conf, fp[n][1], fp['Original'][1]))
            rows.append(r)
        for n in ['Original', 'Retrain-1'] + TEST_NAMES:
            for draw in range(CFG['noise_draws']):
                gen = torch.Generator().manual_seed(9000 + 100 * seed + 10 * draw + bits)
                m, conf = measure(ctx, matched_noise(W[n], Q[n][0], scheme, gen), thr_fp32=fp[n][0]['threshold'])
                r = dict(**base, model=n, **cfg, condition='matched_noise', draw=draw, clip_frac=np.nan, **m)
                if n in TEST_NAMES: r.update(node_recovery(sp, conf, fp[n][1], fp['Original'][1]))
                rows.append(r)
    pd.DataFrame(rows).to_csv(f, index=False)
    log(f'seed {seed}: quantization done in {time.time()-t0:.0f}s ({len(rows)} rows)')

for seed in CFG['evaluation_seeds']: quant_seed(seed)
log('Quantization phase finished.')

## 10. Analysis (Experiment 6 rules, unchanged) and validation verdict

In [ ]:
QD = pd.concat([pd.read_csv(f'{OUT_DIR}/quant/{DS}_seed{s}.csv') for s in CFG['evaluation_seeds']])
k = CFG['min_seeds_agree']; band_fp = GATE['band_fp32']; summary = []
f32 = QD[QD.condition == 'fp32'].pivot_table(index='seed', columns='model', values='auc')
acc32 = QD[QD.condition == 'fp32'].pivot_table(index='seed', columns='model', values='test_acc')
for label, ft in TEST.items():
    gap_fp_ft, gap_fp_o = f32[ft] - f32['Retrain-1'], f32['Original'] - f32['Retrain-1']; dist_fp_ft = gap_fp_ft.abs()
    for (bits, scheme, mode), g in QD[QD.condition != 'fp32'].groupby(['bits', 'scheme', 'scale_mode']):
        q = g[g.condition == 'quantized']; nz = g[g.condition == 'matched_noise']
        a = q.pivot_table(index='seed', columns='model', values='auc'); acc = q.pivot_table(index='seed', columns='model', values='test_acc')
        an = nz.pivot_table(index='seed', columns='model', values='auc')
        band_q = retrain_band(a); eff = max(band_fp, band_q)
        gap_q = a[ft] - a['Retrain-1']; dist_q = gap_q.abs(); dist_q_o = (a['Original'] - a['Retrain-1']).abs()
        d_dist = dist_q - dist_fp_ft; rho = (gap_q - gap_fp_ft) / (gap_fp_o - gap_fp_ft)
        excess = d_dist - ((an[ft] - an['Retrain-1']).abs() - dist_fp_ft)
        qf = q[q.model == ft].set_index('seed'); nr = qf.node_recovery_D - qf.node_recovery_Heval
        dd, ex, nrs, rh, raw = ci95(d_dist), ci95(excess), ci95(nr), ci95(rho), ci95(qf.raw_shift_D_minus_Heval)
        n_unstable = int((qf.unstable_D.astype(bool) | qf.unstable_Heval.astype(bool)).sum())
        drops = {mm: float(acc32[mm].mean() - acc[mm].mean()) for mm in ['Original', 'Retrain-1', ft]}; acc_drop = max(drops.values())
        if acc_drop > CFG['collapse_drop']: verdict = 'NOT INTERPRETABLE: model degraded'
        elif not dist_q_o.mean() > eff: verdict = 'NOT INTERPRETABLE: attack signal lost'
        elif (dist_q.mean() > eff and dd[1] > 0 and (d_dist > 0).sum() >= k and ex[1] > 0 and (excess > 0).sum() >= k):
            verdict = 'REVERSAL, ' + ('specificity not assessable' if n_unstable else
                                      'deleted-node-specific shift' if (nrs[1] > 0 and raw[1] > 0) else 'non-specific shift')
        elif dist_q.mean() <= eff and (dist_q <= eff).sum() >= k: verdict = 'PRESERVED'
        else: verdict = 'INCONCLUSIVE'
        if mode == 'frozen_original' and qf.clip_frac.mean() > 0.01: verdict += ' [clipping-dominated]'
        summary.append(dict(variant=label, model=ft, fp32_gate_passed=GATE['models'][label]['passed'], bits=bits, scheme=scheme,
            scale_mode=mode, verdict=verdict, effective_band=eff, dist_fp32=dist_fp_ft.mean(), dist_q=dist_q.mean(),
            seeds_within_band=f'{int((dist_q <= eff).sum())}/{len(dist_q)}', dist_q_Original=dist_q_o.mean(),
            increase_dist=dd[0], increase_ci_lo=dd[1], increase_ci_hi=dd[2], seeds_increase=f'{int((d_dist > 0).sum())}/{len(d_dist)}',
            reversal_fraction=rh[0], reversal_ci_lo=rh[1], reversal_ci_hi=rh[2], excess_over_noise=ex[0], excess_ci_lo=ex[1],
            seeds_excess=f'{int((excess > 0).sum())}/{len(excess)}', node_recovery_D=qf.node_recovery_D.mean(),
            node_recovery_Heval=qf.node_recovery_Heval.mean(), raw_shift_D=qf.raw_shift_D.mean(), raw_shift_Heval=qf.raw_shift_Heval.mean(),
            raw_shift_D_minus_Heval=raw[0], raw_diff_ci_lo=raw[1], seeds_unstable_recovery=f'{n_unstable}/{len(qf)}',
            acc_drop_max=acc_drop, codes_identical_share=qf.codes_identical_to_Q_original.astype(bool).mean(),
            upd_norm_ratio=qf.upd_norm_ratio.mean(), upd_cosine=qf.upd_cosine.mean(), share_upd_erased=qf.share_upd_erased.mean(),
            med_upd_over_bin=qf.med_upd_over_bin.mean(), clip_frac=qf.clip_frac.mean()))
S = pd.DataFrame(summary); S.to_csv(f'{OUT_DIR}/quantization_summary.csv', index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
for label in TEST:
    print(f'\n=== {TEST[label]} (FP32 gate {"passed" if GATE["models"][label]["passed"] else "NOT passed"}) ===')
    display(S[S.variant == label][['bits', 'scheme', 'scale_mode', 'verdict', 'effective_band', 'dist_fp32', 'dist_q', 'seeds_within_band',
            'reversal_fraction', 'reversal_ci_lo', 'reversal_ci_hi', 'excess_over_noise', 'raw_shift_D', 'raw_shift_Heval',
            'codes_identical_share', 'acc_drop_max', 'upd_cosine', 'share_upd_erased', 'med_upd_over_bin']].round(4))

def row(label, mode='recomputed', bits=CFG['cell_bits']):
    return S[(S.variant == label) & (S.bits == bits) & (S.scheme == 'per_channel') & (S.scale_mode == mode)].iloc[0]
V = {}
PCV, PCB = ('PC-full', 4) if GATE['models']['PC-full']['passed'] or not GATE['models']['PC-full-INT3']['passed'] else ('PC-full-INT3', 3)
V['positive_control_used'] = f'{PCV} (INT{PCB} cells)'
V['V1_PCfull_forgets_in_FP32'] = bool(GATE['models'][PCV]['passed'])
V['V2_PCfull_INT4_reversal_detected'] = bool(all(row(PCV, m, PCB).verdict.startswith('REVERSAL') and
                                                 row(PCV, m, PCB).reversal_ci_lo > CFG['validation_min_reversal_fraction']
                                                 for m in ['recomputed', 'frozen_original']))
V['V3_PCnone_no_reversal'] = bool(not row('PC-none').verdict.startswith('REVERSAL'))
V['V4_FT_replicates_Exp6'] = bool(not row('FT').verdict.startswith('REVERSAL'))
if V['V1_PCfull_forgets_in_FP32'] and V['V2_PCfull_INT4_reversal_detected'] and V['V3_PCnone_no_reversal']:
    reading = 'VALIDATED: the pipeline detects a built-in reversal; Experiment 6 can be read as a real negative result for this setting.'
elif V['V1_PCfull_forgets_in_FP32'] and not V['V2_PCfull_INT4_reversal_detected']:
    reading = 'NOT VALIDATED: built-in reversal not detected; the pipeline lacks power and Experiment 6 must be qualified.'
elif not V['V1_PCfull_forgets_in_FP32']:
    reading = 'POSITIVE CONTROL NOT CONSTRUCTIBLE: PC-full did not forget in FP32; PC-full results are exploratory.'
else:
    reading = 'MIXED: PC-none also shows a reversal; the restrictions themselves create fragility. Investigate before interpreting.'
OUT = dict(validation=V, reading=reading, frozen_doses={v: FROZEN[v]['dose'] for v in CFG['pc_variants']},
           positive_control=[row(PCV, m, PCB).to_dict() for m in ['recomputed', 'frozen_original']],
           pcfull_int4=[row('PC-full', m).to_dict() for m in ['recomputed', 'frozen_original']],
           pcfull_int3_at_int3=[row('PC-full-INT3', m, 3).to_dict() for m in ['recomputed', 'frozen_original']],
           pchalf_int4=row('PC-half').to_dict(), pcnone_int4=row('PC-none').to_dict(), ft_int4=row('FT').to_dict())
json.dump(OUT, open(f'{OUT_DIR}/validation_outcome.json', 'w'), indent=2, default=str)
print('\nValidation:', json.dumps(V, indent=2)); print(reading)
for lab, b in [('PC-full', 4), ('PC-full-INT3', 3), ('PC-half', 4), ('PC-none', 4), ('FT', 4)]:
    r = row(lab, bits=b); print(f"  {lab:12s} INT{b} per-channel recomputed: {r.verdict} | reversal fraction {r.reversal_fraction:.3f} "
                        f"(CI {r.reversal_ci_lo:.3f} to {r.reversal_ci_hi:.3f}) | identical codes {r.codes_identical_share:.0%}")
log('Analysis saved.')

## 11. Plot: reversal fraction by variant and bit-width (per-channel, recomputed scales)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, mode in zip(axes, CFG['scale_modes']):
    for i, lab in enumerate(TEST):
        s = S[(S.variant == lab) & (S.scheme == 'per_channel') & (S.scale_mode == mode)].sort_values('bits', ascending=False)
        x = np.arange(len(s)) + (i - 2) * 0.14
        ax.errorbar(x, s.reversal_fraction, yerr=[s.reversal_fraction - s.reversal_ci_lo, s.reversal_ci_hi - s.reversal_fraction],
                    fmt='o', capsize=3, label=lab)
    ax.set_xticks(range(len(CFG['bits']))); ax.set_xticklabels([f'INT{b}' for b in sorted(CFG['bits'], reverse=True)])
    ax.axhline(0, c='k', lw=0.5); ax.axhline(1, c='grey', ls=':', lw=0.8); ax.set_title(f'per-channel, {mode} scales')
axes[0].set_ylabel('reversal fraction (0 = forgetting kept, 1 = fully undone)'); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig_reversal_fraction.png', dpi=160); plt.show()

## 12. Export everything as a ZIP

In [ ]:
def export():
    base = os.path.basename(OUT_DIR.rstrip('/'))
    shutil.make_archive(base, 'zip', OUT_DIR)
    log(f'ZIP created: {base}.zip ({os.path.getsize(base + ".zip")/1e6:.1f} MB)')
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        from IPython.display import FileLink, display
        print('Kaggle: click the link below, or use the Output panel. In a committed version the ZIP stays in the Output tab.')
        display(FileLink(f'{base}.zip')); return
    try:
        from google.colab import files; files.download(f'{base}.zip')
    except Exception:
        print('Download the ZIP from the file panel on the left.')
export()

### Emergency export
If the session is about to disconnect, run only the cell below.

In [ ]:
export()